In [ ]:
import os
import certifi
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests 
from langchain_tavily import TavilySearch

In [2]:
from langchain.agents import create_react_agent, AgentExecutor

In [3]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================

os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [5]:
search_tool = TavilySearchResults(max_results=2)

In [6]:
result = search_tool.invoke("Give me the latest news on AI")
result

[{'title': 'AI News | Latest News | Insights Powering AI-Driven Business ...',
  'url': 'https://www.artificialintelligence-news.com',
  'content': 'Explore More\n\n# Hershey applies AI across its supply chain operations\n\n# With higher risk and broader impact, AI security is the new supply chain problem\n\n# Gatik raises $200M to scale AI-powered autonomous freight\n\n# OneRail uses Nvidia AI for real-time last-mile delivery optimisation\n\n#### Applications\n\n### Thailand becomes one of the first in Asia to get the Sora app\n\nEntertainment & Media\n\nOctober 30, 2025\n\n### Malaysia launches Ryt Bank, its first AI-powered bank\n\nFinance AI\n\nAugust 26, 2025\n\n### Google’s Veo 3 AI video creation tools are now widely available\n\nAI in Action\n\nJuly 29, 2025\n\n#### Computer Vision\n\n### Microsoft’s Majorana 2 quantum chip is also a case study for agentic AI in R&D\n\nInside AI\n\nJune 3, 2026\n\n### US and Japan announce sweeping AI and tech collaboration [...] Artificial Int

In [53]:
# ==========================================
# LLM
# ==========================================

# llm = ChatGroq(
#     model="openai/gpt-oss-120b",
#     temperature=0,
#     api_key=GROQ_API_KEY
# )



# ==========================================
# LLM
# ==========================================

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key=GROQ_API_KEY
)

In [54]:
response = llm.invoke("what year is it?")
print(response)

NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.1-8b-instant` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [47]:
# from langsmith import Client

# client = Client()

# prompt = client.pull_prompt(
#     "hwchase17/react",
#     dangerously_pull_public_prompt=True
# )




# using the manual one:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# ==========================================
# PROMPT
# ==========================================

# prompt = ChatPromptTemplate.from_messages([
#     (
#         "system",
#         "You are a helpful AI assistant. Use the available tools when necessary "
#         "to answer the user's questions."
#     ),
#     ("human", "{input}"),
#     MessagesPlaceholder(variable_name="agent_scratchpad"),
# ])




from langchain_core.prompts import PromptTemplate

# ==========================================
# MANUAL REACT PROMPT
# ==========================================

prompt = PromptTemplate.from_template("""
Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}
""")

In [48]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], input_types={}, partial_variables={}, template='\nAnswer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}\n')

In [49]:
# ==========================================
# TOOLS
# ==========================================

tools = [search_tool]    # we can add multiple tools here if we want, and its a dictionary of tool name that we can add here.

In [50]:
# ==========================================
# CREATE AGENT
# ==========================================

# agent = create_react_agent(
#     llm=llm,
#     tools=tools,
#     prompt=prompt
# )


# newer one:
# from langchain.agents import create_tool_calling_agent

# # ==========================================
# # CREATE AGENT
# # ==========================================

# agent = create_tool_calling_agent(
#     llm=llm,
#     tools=tools,
#     prompt=prompt
# )




from langchain.agents import create_react_agent

# ==========================================
# CREATE AGENT
# ==========================================

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [51]:
# ==========================================
# EXECUTOR
# ==========================================

# agent_executor = AgentExecutor(
#     agent=agent,
#     tools=tools,
#     verbose=True
# )



from langchain.agents import AgentExecutor

# ==========================================
# EXECUTOR
# ==========================================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [52]:
# ==========================================
# RUN
# ==========================================

# response = agent_executor.invoke({
#     "input": "Tell me the latest news related to the Rasuwa Nepal flood"
# })

# print(response["output"])



# ==========================================
# RUN
# ==========================================

response = agent_executor.invoke({
    "input": "Tell me the latest news related to the Rasuwa Nepal flood"
})

print(response["output"])



> Entering new AgentExecutor chain...


APIError: Tool choice is none, but model called a tool

In [ ]:
# THIS IS THE MODERN APPROACH:



# ============================================================
# MODERN TOOL-CALLING AGENT
# Groq + LangChain + Tavily
# ============================================================

# ------------------------------------------------------------
# 1. IMPORTS
# ------------------------------------------------------------

import os
import certifi

from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

from langchain.agents import create_tool_calling_agent, AgentExecutor


# ------------------------------------------------------------
# 2. LOAD ENVIRONMENT VARIABLES
# ------------------------------------------------------------

# Fix SSL certificate issues on some Windows/Python setups
os.environ["SSL_CERT_FILE"] = certifi.where()

# Load variables from .env
load_dotenv()

# Get API keys
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")


# ------------------------------------------------------------
# 3. CHECK API KEYS
# ------------------------------------------------------------

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is missing from .env")

if not TAVILY_API_KEY:
    raise ValueError("TAVILY_API_KEY is missing from .env")


# ------------------------------------------------------------
# 4. CREATE GROQ LLM
# ------------------------------------------------------------

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=GROQ_API_KEY
)


# ------------------------------------------------------------
# 5. CREATE TAVILY SEARCH TOOL
# ------------------------------------------------------------

search_tool = TavilySearch(
    max_results=2
)

# Put all tools into a list
tools = [search_tool]


# ------------------------------------------------------------
# 6. CREATE MODERN TOOL-CALLING PROMPT
# ------------------------------------------------------------

prompt = ChatPromptTemplate.from_messages([

    # System instructions
    (
        "system",
        """
        You are a helpful AI assistant.

        You have access to tools that you can use when necessary.

        Use the available tools when the user's question requires
        current information, web searches, or external information.

        If you use a tool, use the result from that tool to formulate
        your final answer.

        Give clear and accurate answers.
        """
    ),

    # User's question
    (
        "human",
        "{input}"
    ),

    # This is where LangChain stores the agent's
    # previous tool calls and tool results.
    MessagesPlaceholder(
        variable_name="agent_scratchpad"
    ),
])


# ------------------------------------------------------------
# 7. CREATE TOOL-CALLING AGENT
# ------------------------------------------------------------

agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)


# ------------------------------------------------------------
# 8. CREATE AGENT EXECUTOR
# ------------------------------------------------------------

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,

    # Shows what the agent is doing in the terminal/notebook
    verbose=True
)


# ------------------------------------------------------------
# 9. TEST THE AGENT
# ------------------------------------------------------------

response = agent_executor.invoke({
    "input": "What is the latest news about the Rasuwa Nepal flood?"
})


# ------------------------------------------------------------
# 10. PRINT FINAL ANSWER
# ------------------------------------------------------------

print("\n================ FINAL ANSWER ================\n")

print(response["output"])



> Entering new AgentExecutor chain...

Invoking: `tavily_search` with `{'query': 'Rasuwa Nepal flood latest news', 'search_depth': 'advanced', 'time_range': 'day', 'topic': 'news'}`


{'query': 'Rasuwa Nepal flood latest news', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.tribuneindia.com/news/world/nepal-flood-death-toll-reaches-1385-as-authorities-continue-search-relief-efforts/amp', 'title': 'Nepal flood death toll reaches 1,385 as authorities continue search, relief efforts', 'content': 'Updated At : 02:07 PM Sep 11, 2026 IST\n\n \n\nAdvertisement\n\nKathmandu [Nepal], September 11 (ANI): The death toll from the Rasuwa floods has reached 1,385, while 5,130 people remain missing, according to the latest Search, Rescue and Relief update issued by Nepal’s National Disaster Risk Reduction and Management Authority (NDRRMA).\n\nAdvertisement\n\nThe update, issued at 11:00 am on Friday, reported casualties across several districts. Chitwan 

# IMPORTANT NOTE: READ THIS TO KNOW THE ACTUAL DIFFERENCE BETWEEN THE OLD TRADITIONAL REACT TOOL CALLING AND MODERN APPROACT TO CALL TOOL:

In [ ]:
# # AGENTS, REACT & MODERN TOOL CALLING
# # ===================================

# ## 1. WHAT IS AN AI AGENT?

# An AI Agent is an LLM-based system that can:

# 1. Understand the user's request
# 2. Decide what action is needed
# 3. Use tools when necessary
# 4. Receive the tool result
# 5. Continue reasoning/processing
# 6. Give the final answer

# Basic idea:

# User
#   ↓
# LLM
#   ↓
# Does it need a tool?
#   ├── NO  → Final Answer
#   │
#   └── YES
#         ↓
#       Tool
#         ↓
#     Tool Result
#         ↓
#        LLM
#         ↓
#    Final Answer


# ==================================================
# 2. WHAT IS A TOOL?
# ==================================================

# A tool is an external function/capability that an LLM can use.

# Examples:

# - Web search
# - Calculator
# - Weather API
# - Database
# - Python/code execution
# - File search
# - Sending an email
# - Calling another API

# In our project:

# Groq = LLM
# Tavily = Tool for web search
# LangChain = Framework connecting the LLM and tools


# ==================================================
# 3. OUR CURRENT PROJECT
# ==================================================

# We are using:

#     Groq
#       +
#     LangChain
#       +
#     Tavily
#       =
#     AI Agent with Web Search


# Architecture:

#               USER
#                 │
#                 ▼
#           ┌───────────┐
#           │  Groq LLM │
#           └─────┬─────┘
#                 │
#         Need external information?
#              /       \
#            NO         YES
#            │           │
#            │           ▼
#            │      ┌─────────┐
#            │      │ Tavily  │
#            │      │ Search  │
#            │      └────┬────┘
#            │           │
#            │           ▼
#            │      Search Result
#            │           │
#            └─────┬─────┘
#                  ▼
#            Final Answer


# ==================================================
# 4. WHAT IS REACT?
# ==================================================

# ReAct = Reason + Act

# It is an agent architecture/pattern where the LLM
# follows a reasoning and action loop.

# The classic ReAct format looks like:

# Question
#    ↓
# Thought
#    ↓
# Action
#    ↓
# Action Input
#    ↓
# Observation
#    ↓
# Thought
#    ↓
# Action
#    ↓
# Observation
#    ↓
# ...
#    ↓
# Final Answer


# Example:

# Question:
# "What is the latest news about Rasuwa flood?"

# Thought:
# I need current information, so I should search the web.

# Action:
# tavily_search

# Action Input:
# "Rasuwa Nepal flood latest news"

# Observation:
# [Tavily search results]

# Thought:
# I have enough information to answer.

# Final Answer:
# [Answer based on search results]


# ==================================================
# 5. REACT PROMPT
# ==================================================

# The classic ReAct prompt explicitly teaches the LLM
# a format such as:

# Question: the input question you must answer

# Thought: you should always think about what to do

# Action: the action to take

# Action Input: the input to the action

# Observation: the result of the action

# Thought: I now know the final answer

# Final Answer: the final answer to the original input


# The prompt also contains:

# {tools}

# {tool_names}

# {input}

# {agent_scratchpad}


# The purpose is to tell the LLM:

# "Here are your tools. Decide which one to use and
# follow this Thought → Action → Observation format."


# ==================================================
# 6. LANGCHAIN CLASSIC REACT AGENT
# ==================================================

# The LangChain function used in the tutorial is:

# create_react_agent()


# Conceptually:

# LLM
#  +
# ReAct Prompt
#  +
# Tools
#  ↓
# create_react_agent()
#  ↓
# ReAct Agent
#  ↓
# AgentExecutor


# Example:

# agent = create_react_agent(
#     llm=llm,
#     tools=tools,
#     prompt=prompt
# )


# ==================================================
# 7. IMPORTANT PROBLEM WE ENCOUNTERED
# ==================================================

# We initially used:

# hub.pull("hwchase17/react")


# This was the standard way to obtain the ReAct prompt
# from LangChain Hub.

# However, our newer LangChain/LangSmith setup blocked
# public prompt pulling because of security restrictions.

# Therefore, instead of depending on the Hub prompt,
# we manually recreated the ReAct prompt.

# Manual prompt:

# prompt = PromptTemplate.from_template("""
# Answer the following questions as best you can.
# You have access to the following tools:

# {tools}

# Use the following format:

# Question: the input question you must answer
# Thought: you should always think about what to do
# Action: the action to take, should be one of [{tool_names}]
# Action Input: the input to the action
# Observation: the result of the action
# ... (this Thought/Action/Action Input/Observation can repeat N times)
# Thought: I now know the final answer
# Final Answer: the final answer to the original input question

# Begin!

# Question: {input}
# Thought:{agent_scratchpad}
# """)


# ==================================================
# 8. IMPORTANT ERROR WE ENCOUNTERED
# ==================================================

# With the ReAct setup and Groq model we encountered:

# APIError:
# Tool choice is none, but model called a tool


# This does NOT mean:

# "Groq cannot use tools."


# It means that the particular model/provider + LangChain
# ReAct configuration had a compatibility issue with how
# the tool invocation was being handled.

# Groq DOES support tool calling.

# Therefore:

# Groq ≠ incapable of tool calling

# The issue was specifically with that ReAct configuration.


# ==================================================
# 9. MODERN TOOL CALLING
# ==================================================

# Modern tool calling works differently from classic ReAct.

# Instead of making the LLM generate text like:

# Thought:
# Action:
# Action Input:
# Observation:


# The LLM can return a structured tool call.

# Example conceptually:

# {
#     "name": "tavily_search",
#     "arguments": {
#         "query": "Rasuwa Nepal flood latest news"
#     }
# }


# LangChain then executes the tool.

# Architecture:

# USER
#   ↓
# GROQ LLM
#   ↓
# Structured Tool Call
#   ↓
# TAVILY
#   ↓
# Tool Result
#   ↓
# GROQ LLM
#   ↓
# FINAL ANSWER


# The model does not need to manually write:

# Action: tavily_search


# The tool call is structured.


# ==================================================
# 10. MODERN TOOL CALLING IN LANGCHAIN
# ==================================================

# For modern tool calling, LangChain provides:

# create_tool_calling_agent()


# We used:

# agent = create_tool_calling_agent(
#     llm=llm,
#     tools=tools,
#     prompt=prompt
# )


# And:

# agent_executor = AgentExecutor(
#     agent=agent,
#     tools=tools,
#     verbose=True
# )


# ==================================================
# 11. MODERN TOOL-CALLING PROMPT
# ==================================================

# The prompt is different from the classic ReAct prompt.

# We used:

# prompt = ChatPromptTemplate.from_messages([

#     (
#         "system",
#         """
#         You are a helpful AI assistant.

#         You have access to tools that you can use when necessary.

#         Use the available tools when the user's question requires
#         current information, web searches, or external information.

#         If you use a tool, use the result from that tool to formulate
#         your final answer.

#         Give clear and accurate answers.
#         """
#     ),

#     ("human", "{input}"),

#     MessagesPlaceholder(
#         variable_name="agent_scratchpad"
#     ),
# ])


# IMPORTANT:

# The modern tool-calling prompt does NOT need the old:

# {tools}

# {tool_names}

# Thought:
# Action:
# Action Input:
# Observation:


# format.


# ==================================================
# 12. OUR SUCCESSFUL MODERN TOOL-CALLING TEST
# ==================================================

# We used:

# Groq
# +
# openai/gpt-oss-20b
# +
# Tavily
# +
# LangChain
# +
# create_tool_calling_agent()


# The LLM was able to call Tavily.

# The AgentExecutor output showed:

# > Entering new AgentExecutor chain...

# Invoking: `tavily_search`

# with:

# {
#     'query': 'Rasuwa Nepal flood latest news',
#     ...
# }


# THIS IS PROOF THAT THE TOOL WAS ACTUALLY USED.


# ==================================================
# 13. HOW WE KNOW A TOOL WAS USED
# ==================================================

# When verbose=True, we saw:

# Invoking: `tavily_search`

# This means:

# 1. User asked a question
# 2. LLM determined that web information was required
# 3. LLM generated a structured tool call
# 4. LangChain received the tool call
# 5. LangChain executed Tavily
# 6. Tavily returned search results
# 7. The agent used those results
# 8. The LLM generated the final answer


# So this:

# Invoking: `tavily_search`

# is NOT just normal LLM output.

# It represents an actual tool execution.


# ==================================================
# 14. REACT VS MODERN TOOL CALLING
# ==================================================

#                 AGENT TOOL USE
#                       │
#           ┌───────────┴───────────┐
#           │                       │
#         ReAct              Modern Tool Calling
#           │                       │
#           ▼                       ▼
#    Text-based format       Structured tool call
#           │                       │
#           ▼                       ▼
#       Thought                  Tool Call
#           ↓                       ↓
#        Action                   Tool
#           ↓                       ↓
#     Action Input             Tool Result
#           ↓                       ↓
#     Observation                   ↓
#           ↓                       ↓
#        Thought                    LLM
#           ↓                       ↓
#     Final Answer             Final Answer


# ==================================================
# 15. SIMPLE COMPARISON
# ==================================================

# REACT:

# LLM outputs something like:

# Thought:
# I need to search.

# Action:
# tavily_search

# Action Input:
# "Rasuwa flood latest news"

# Observation:
# [search result]

# Thought:
# Now I can answer.

# Final Answer:
# ...


# MODERN TOOL CALLING:

# LLM produces:

# Tool Call:
# tavily_search(
#     query="Rasuwa flood latest news"
# )

# Tavily:
# [search result]

# LLM:
# Final Answer:
# ...


# ==================================================
# 16. WHY IS THE TUTOR USING REACT?
# ==================================================

# The tutor is not necessarily wrong.

# ReAct is an important foundational agent architecture.

# It makes the agent loop very easy to understand:

# Reason
#   ↓
# Act
#   ↓
# Observe
#   ↓
# Reason
#   ↓
# Act
#   ↓
# Observe
#   ↓
# Final Answer


# ReAct is useful for learning:

# - What an agent is
# - How an agent decides to use a tool
# - The reasoning/action loop
# - Tool execution
# - Observations
# - Multiple tool calls
# - AgentExecutor
# - Agent scratchpad


# Modern tool calling is more practical for many
# current applications because the model can produce
# structured tool calls.


# ==================================================
# 17. VERY IMPORTANT:
#    REACT IS NOT THE SAME AS TOOL CALLING
# ==================================================

# ReAct is an agent reasoning/action pattern.

# Tool calling is a mechanism/interface through which
# an LLM can request a tool.

# They can work together, but they are not identical concepts.


# Think:

# ReAct = HOW the agent loop is structured

# Tool Calling = HOW the model communicates a request
#                to execute a tool


# ==================================================
# 18. WHY WE SHOULD LEARN BOTH
# ==================================================

# For AI Engineer / Agentic AI learning:

# FIRST:

# Learn ReAct

# Understand:

# User
#  ↓
# Reason
#  ↓
# Action
#  ↓
# Tool
#  ↓
# Observation
#  ↓
# Reason
#  ↓
# Final Answer


# THEN:

# Learn modern structured tool calling

# Understand:

# User
#  ↓
# LLM
#  ↓
# Structured Tool Call
#  ↓
# Tool
#  ↓
# Tool Result
#  ↓
# LLM
#  ↓
# Final Answer


# THEN:

# Learn more advanced agent architectures:

# LangChain Agents
#        ↓
# Multiple Tools
#        ↓
# RAG + Agents
#        ↓
# Memory
#        ↓
# LangGraph
#        ↓
# Multi-Agent Systems
#        ↓
# Real Agentic AI Applications


# ==================================================
# 19. OUR CURRENT STACK
# ==================================================

# LLM:

# ChatGroq


# Current model tested:

# openai/gpt-oss-20b


# Tool:

# TavilySearch


# Framework:

# LangChain


# Modern agent:

# create_tool_calling_agent()


# Executor:

# AgentExecutor()


# Environment variables:

# GROQ_API_KEY
# TAVILY_API_KEY


# ==================================================
# 20. CURRENT MODERN AGENT ARCHITECTURE
# ==================================================

#                     USER
#                       │
#                       ▼
#               ┌──────────────┐
#               │   LangChain  │
#               │    Agent     │
#               └──────┬───────┘
#                      │
#                      ▼
#               ┌──────────────┐
#               │   Groq LLM   │
#               │ GPT-OSS-20B  │
#               └──────┬───────┘
#                      │
#              Does it need a tool?
#                  /          \
#                NO            YES
#                │              │
#                │              ▼
#                │       Structured Tool Call
#                │              │
#                │              ▼
#                │       ┌─────────────┐
#                │       │   Tavily    │
#                │       │   Search    │
#                │       └──────┬──────┘
#                │              │
#                │              ▼
#                │         Tool Result
#                │              │
#                └──────┬───────┘
#                       ▼
#                  Groq LLM
#                       │
#                       ▼
#                 FINAL ANSWER


# ==================================================
# 21. MOST IMPORTANT THINGS TO REMEMBER
# ==================================================

# 1. An LLM by itself is not automatically an agent.

# 2. A tool gives the LLM an external capability.

# 3. An agent can decide when to use a tool.

# 4. ReAct is one way to structure an agent.

# 5. Classic ReAct uses a textual format such as:

#    Thought → Action → Observation

# 6. Modern tool calling uses structured tool calls.

# 7. Groq supports tool calling.

# 8. Tavily is our web-search tool.

# 9. LangChain connects the LLM, tools, and agent.

# 10. create_react_agent() is for the classic ReAct approach.

# 11. create_tool_calling_agent() is for modern
#     tool-calling agents.

# 12. The error:

#     "Tool choice is none, but model called a tool"

#     does NOT mean Groq cannot use tools.

#     It indicates a compatibility/configuration issue
#     with that particular ReAct + model setup.

# 13. Our modern Groq + Tavily test successfully called:

#     tavily_search

# 14. Seeing:

#     Invoking: `tavily_search`

#     confirms that the agent actually executed the tool.

# 15. ReAct is still worth learning because it teaches
#     the fundamental agent loop.

# 16. Modern tool calling is important for building
#     practical modern agents.


# ==================================================
# 22. FINAL BIG PICTURE
# ==================================================

#              AI AGENT
#                 │
#                 ▼
#           ┌───────────┐
#           │    LLM    │
#           └─────┬─────┘
#                 │
#         Decide what to do
#                 │
#         ┌───────┴────────┐
#         │                │
#      No Tool           Tool Needed
#         │                │
#         ▼                ▼
#    Final Answer     Tool Call
#                          │
#                          ▼
#                        Tool
#                          │
#                          ▼
#                     Tool Result
#                          │
#                          ▼
#                         LLM
#                          │
#                          ▼
#                    Final Answer


# There are different ways to implement the agent loop.

# One important traditional approach:

#              REACT
#                │
#        Thought → Action
#                ↓
#           Observation
#                ↓
#             Thought
#                ↓
#           Final Answer


# One important modern approach:

#         TOOL CALLING
#                │
#           Structured
#           Tool Call
#                ↓
#               Tool
#                ↓
#           Tool Result
#                ↓
#               LLM
#                ↓
#           Final Answer


# Both are important concepts.

# ReAct helps understand the FUNDAMENTALS.

# Modern tool calling helps build PRACTICAL MODERN
# tool-using agents.

# ==================================================

In [ ]:
# ReAct makes the process visible

# With classic ReAct, you explicitly teach the model:

# Thought: I need current information.
# Action: tavily_search
# Action Input: "Rasuwa flood latest news"
# Observation: [results]

# Thought: I have enough information.
# Final Answer: ...

# So you can see the agent's reasoning loop.

#              REACT
#                │
#                ▼
#         ┌─────────────┐
#         │   THOUGHT   │
#         │ What should │
#         │ I do?       │
#         └──────┬──────┘
#                ↓
#         ┌─────────────┐
#         │   ACTION    │
#         │ Use Tavily  │
#         └──────┬──────┘
#                ↓
#         ┌─────────────┐
#         │ OBSERVATION │
#         │ Tool result │
#         └──────┬──────┘
#                ↓
#         ┌─────────────┐
#         │   THOUGHT   │
#         │ What now?   │
#         └──────┬──────┘
#                ↓
#           Final Answer
# Modern tool calling does the same basic thing

# Suppose you ask:

# "What's the latest news about Rasuwa?"

# The model internally determines:

# "I need current information, so I should use the search tool."

# Then it produces a structured tool call:

# tool_call:
#     name = tavily_search
#     arguments = {
#         query: "Rasuwa Nepal flood latest news"
#     }

# Tavily returns:

# tool_result:
#     [news articles...]

# Then the model processes the result and decides:

# "I have enough information. I'll answer."

# Then:

# Final Answer

# So conceptually:

#         MODERN TOOL CALLING
#                 │
#                 ▼
#         ┌──────────────┐
#         │ Understand   │
#         │ the question │
#         └──────┬───────┘
#                ↓
#         ┌──────────────┐
#         │ Decide what  │
#         │ to do        │
#         └──────┬───────┘
#                ↓
#           Need a tool?
#            /       \
#          NO         YES
#          │           │
#          │           ▼
#          │       Tool Call
#          │           ↓
#          │          Tool
#          │           ↓
#          │      Tool Result
#          │           ↓
#          └─────►  LLM
#                    │
#                    ▼
#               Final Answer
# So where did the "Thought" go?

# It isn't necessarily exposed as a Thought: text field.

# That's the key.

# ReAct says:

# Thought: ...
# Action: ...
# Observation: ...

# Modern tool calling says, essentially:

# "Here is the tool I want to call,
# with these arguments."

# The model's internal reasoning/decision process isn't the same thing as the visible Thought: field.

# Think of it this way

# Imagine you're asking a human assistant:

# "What's today's weather?"

# ReAct-style explanation

# The assistant writes down:

# THOUGHT:
# I need today's weather.

# ACTION:
# Call weather API.

# OBSERVATION:
# 25°C and cloudy.

# THOUGHT:
# Now I can answer.

# ANSWER:
# It's 25°C and cloudy.
# Modern tool-calling style

# The assistant simply tells the system:

# CALL WEATHER API
# city = Kathmandu

# The system gives the result:

# 25°C, cloudy

# Then the assistant says:

# It's 25°C and cloudy.

# The assistant still had to decide that it needed the weather API.

# You just aren't forcing it to write:

# Thought:
# Action:
# Observation: